# scANVI label transfer for AON snRNA-seq (supplementary, GPU)

Probabilistic label transfer of ABC **WMB-10Xv2-OLF** subclass labels onto the 32,952 in-house AON nuclei, with per-cell posterior probabilities so low-confidence cells (the leiden-4 GABAergic cluster) are flagged as unassignable rather than force-mapped. This complements the correlation-based label transfer in `06_reference_concordance.ipynb` and produces the data behind the reference-concordance figure.

The step needs a GPU and is run out-of-band, so it is not part of the Snakemake DAG. The `scvi-tools` dependency is declared in `environment.yml`.

**Input** `data/aon_10x/scanvi_input.h5ad`: the in-house preprocessed nuclei (query) concatenated with the ABC OLF reference cells over shared genes, raw integer counts in `.X`, and obs columns `batch` (`reference` or `query`), `subclass` (ABC labels, `Unknown` for query), `leiden`, and `lineage`.

**Outputs** `results/tables/scanvi_predictions.csv` (per-query-cell call and posterior) and `results/tables/scanvi_cluster_summary.csv` (per-cluster confidence).

In [ ]:
import pandas as pd
import scanpy as sc
import scvi

scvi.settings.seed = 0
print("scvi-tools", scvi.__version__)

adata = sc.read_h5ad("data/aon_10x/scanvi_input.h5ad")
adata.layers["counts"] = adata.X.copy()          # raw integer counts
print(adata)
print(adata.obs["batch"].value_counts())

In [ ]:
# Train scVI on the shared reference/query latent space
scvi.model.SCVI.setup_anndata(adata, layer="counts", batch_key="batch")
scvi_model = scvi.model.SCVI(adata, n_layers=2, n_latent=30, gene_likelihood="nb")
scvi_model.train(max_epochs=200, early_stopping=True)

In [ ]:
# scANVI semi-supervised step, reference labelled and query set to Unknown
lvae = scvi.model.SCANVI.from_scvi_model(
    scvi_model, adata=adata, labels_key="subclass", unlabeled_category="Unknown")
lvae.train(max_epochs=20, n_samples_per_label=100)

In [ ]:
# Predict labels and per-cell posterior probabilities
adata.obs["scanvi_pred"] = lvae.predict()
soft = lvae.predict(soft=True)                   # cells x subclasses
adata.obs["scanvi_maxprob"] = soft.max(axis=1).values

In [ ]:
# Export query-cell predictions
q = adata.obs[adata.obs["batch"] == "query"].copy()
soft_q = soft.loc[q.index]
top3 = soft_q.apply(lambda r: ";".join(f"{c}:{r[c]:.3f}" for c in r.nlargest(3).index), axis=1)
out = pd.DataFrame({
    "barcode":        [i.replace("query_", "") for i in q.index],
    "leiden":         q["leiden"].values,
    "lineage":        q["lineage"].values,
    "scanvi_pred":    q["scanvi_pred"].values,
    "scanvi_maxprob": q["scanvi_maxprob"].values,
    "top3":           top3.values,
})
out.to_csv("results/tables/scanvi_predictions.csv", index=False)
print("wrote scanvi_predictions.csv", out.shape)

In [ ]:
# Per-cluster confidence summary
g = out.groupby("leiden").agg(
    n=("barcode", "size"),
    inhouse_lineage=("lineage", lambda s: s.value_counts().index[0]),
    scanvi_mean_conf=("scanvi_maxprob", "mean"),
    scanvi_med_conf=("scanvi_maxprob", "median"),
    scanvi_top_subclass=("scanvi_pred", lambda s: s.value_counts().index[0]),
    scanvi_top_frac=("scanvi_pred", lambda s: s.value_counts(normalize=True).iloc[0]),
).sort_index(key=lambda x: x.astype(int))
g.to_csv("results/tables/scanvi_cluster_summary.csv")
print(g.to_string())